# German ASR Pipeline - Data Exploration

This notebook demonstrates how to explore and visualize the dataset.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from asr_pipeline.utils import load_manifest, compute_dataset_stats, print_dataset_stats

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Manifest

In [2]:
# Load the manifest
manifest_path = '../manifests/03_preprocessed_manifest.parquet'

if Path(manifest_path).exists():
    df = load_manifest(manifest_path)
    print(f"Loaded {len(df)} samples")
    print(f"\nColumns: {list(df.columns)}")
else:
    print(f"Manifest not found at {manifest_path}")
    print("Please run the pipeline first to generate the manifest.")

Manifest not found at ../manifests/03_preprocessed_manifest.parquet
Please run the pipeline first to generate the manifest.


## Dataset Statistics

In [3]:
if 'df' in locals():
    print_dataset_stats(df, "Dataset Overview")

## Duration Distribution

In [4]:
if 'df' in locals() and 'duration_sec' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    sns.histplot(df['duration_sec'], bins=50, kde=True, ax=axes[0])
    axes[0].set_xlabel('Duration (seconds)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Duration Distribution')
    
    # Box plot
    sns.boxplot(y=df['duration_sec'], ax=axes[1])
    axes[1].set_ylabel('Duration (seconds)')
    axes[1].set_title('Duration Box Plot')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nDuration Statistics:")
    print(df['duration_sec'].describe())

## Split Distribution

In [5]:
if 'df' in locals() and 'split' in df.columns:
    split_counts = df['split'].value_counts()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    split_counts.plot(kind='bar', ax=ax)
    ax.set_xlabel('Split')
    ax.set_ylabel('Count')
    ax.set_title('Split Distribution')
    ax.tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nSplit Counts:")
    print(split_counts)

## Domain Distribution

In [6]:
if 'df' in locals() and 'domain' in df.columns:
    domain_counts = df['domain'].value_counts()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    domain_counts.plot(kind='bar', ax=ax)
    ax.set_xlabel('Domain')
    ax.set_ylabel('Count')
    ax.set_title('Domain Distribution')
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nDomain Counts:")
    print(domain_counts)

## Sample Transcripts

In [7]:
if 'df' in locals():
    transcript_col = None
    if 'transcript_normalized' in df.columns:
        transcript_col = 'transcript_normalized'
    elif 'transcript_raw' in df.columns:
        transcript_col = 'transcript_raw'
    
    if transcript_col:
        print(f"\nSample Transcripts ({transcript_col}):")
        print("=" * 80)
        for idx, row in df.head(5).iterrows():
            print(f"\nSample {row.get('sample_id', idx)}:")
            print(f"  Duration: {row.get('duration_sec', 'N/A')}s")
            print(f"  Transcript: {row[transcript_col][:200]}...")

## Validation Status

In [8]:
if 'df' in locals() and 'is_valid' in df.columns:
    valid_count = df['is_valid'].sum()
    invalid_count = len(df) - valid_count
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(
        [valid_count, invalid_count],
        labels=['Valid', 'Invalid'],
        autopct='%1.1f%%',
        colors=['#2ecc71', '#e74c3c']
    )
    ax.set_title('Validation Status')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nValid: {valid_count} ({valid_count/len(df)*100:.1f}%)")
    print(f"Invalid: {invalid_count} ({invalid_count/len(df)*100:.1f}%)")
    
    if invalid_count > 0 and 'validation_issues' in df.columns:
        print(f"\nTop Issues:")
        issues = df[~df['is_valid']]['validation_issues'].value_counts().head(5)
        print(issues)

## Export Summary

In [9]:
if 'df' in locals():
    # Save summary statistics
    stats = compute_dataset_stats(df)
    
    summary_df = pd.DataFrame([{
        'Metric': k,
        'Value': v if not isinstance(v, dict) else str(v)
    } for k, v in stats.items()])
    
    output_path = '../artifacts/data_summary.csv'
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(output_path, index=False)
    print(f"Summary saved to {output_path}")